# HW3 – Classification: Best Investment Strategy per Stock

**Goal:** predict which of three investment strategies works best for a stock, using its sector, industry and risk/return ratios.

* Classes (`Best_Strategy`): `DCA_Buy_Hold`, `Drawdown60`, `Year_Sept_April`
* Models compared: MLP, Deep Learning net (PyTorch) and XGBoost
* Output: an **integer class** (0, 1 or 2) obtained with `torch.max(y_pred, dim=1)` (argmax), not a probability
* Metrics: accuracy, precision, recall, F1-measure, confusion matrix
* Deployment: ONNX model + GitHub Pages (`onnxruntime-web`)

## Libraries

In [ ]:
!pip -q install onnx onnxruntime onnxmltools onnxscript openpyxl mlxtend

In [ ]:
import numpy as np
import torch
import pandas as pd
import sklearn
import random
import json
import os

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
import matplotlib.pyplot as plt

from mlxtend.plotting import heatmap
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

In [ ]:
import onnxruntime as rt
import onnx
import onnxmltools

from onnxmltools.convert.common.data_types import FloatTensorType   ## (skl2onnx version is rejected by newer onnxmltools)

## Parameters

In [ ]:
## 0.001, 0.0001, 0.0003, 0.01, 0.03

batch_size    = 16
learning_rate = 0.003 ## 0.001
N_Epochs      = 1000  ## increase if needed

epsilon = 0.0001

random_seed = 42      ## fixed so the report results can be reproduced
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

## Read data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_data = '/content/drive/MyDrive/HW3/Raw Data_3.xlsx'

stock_raw_data = pd.read_excel( path_data )

In [ ]:
stock_raw_data

## Encode categorical columns

`Sector`, `Industry` and the target `Best_Strategy` are text, so they are mapped to integer IDs (same idea as `species.map(...)` in the Iris example).

In [ ]:
sector_names   = sorted( stock_raw_data['Sector'].dropna().unique() )
industry_names = sorted( stock_raw_data['Industry'].dropna().unique() )
strategy_names = sorted( stock_raw_data['Best_Strategy'].dropna().unique() )

sector_map   = { name: i for i, name in enumerate(sector_names)   }
industry_map = { name: i for i, name in enumerate(industry_names) }
strategy_map = { name: i for i, name in enumerate(strategy_names) }

print('Sector map:',   sector_map)
print('Industry map:', industry_map)
print('Strategy map:', strategy_map)

stock_raw_data['Sector']        = stock_raw_data['Sector'].map(sector_map)
stock_raw_data['Industry']      = stock_raw_data['Industry'].map(industry_map)
stock_raw_data['Best_Strategy'] = stock_raw_data['Best_Strategy'].map(strategy_map)

In [ ]:
stock_raw_data

## Select features and target

In [ ]:
stock_data = stock_raw_data.copy()

## Best_Strategy is the target, Symbol is only an identifier
feature_columns = [ c for c in stock_data.columns if c not in ['Symbol', 'Best_Strategy'] ]

for col in feature_columns:
    stock_data[col] = pd.to_numeric( stock_data[col], errors='coerce' )

stock_data = stock_data.dropna( subset=['Best_Strategy'] ).reset_index(drop=True)

headers_list = feature_columns + ['Best_Strategy']

n_features = len(feature_columns)
n_classes  = len(strategy_names)

print('Input columns:', feature_columns)
print('Number of features:', n_features)
print('Number of classes:',  n_classes)

In [ ]:
headers_list

## Data Analysis

* Correlation matrix

Note: `Sector` and `Industry` are arbitrary integer IDs, so their Pearson correlations should be interpreted with caution.

In [ ]:
cm_data = stock_data[headers_list].fillna( stock_data[headers_list].median() )

cm = np.corrcoef(   cm_data.values.T   )
hm = heatmap(cm, row_names=headers_list, column_names=headers_list, figsize=(20,10))
plt.show()

In [ ]:
## correlation of each feature with the target
pd.Series( cm[-1, :-1], index=feature_columns ).sort_values()

## Process Data

In [ ]:
## Convert Pandas to Numpy

X = stock_data[feature_columns].to_numpy()

y = stock_data[['Best_Strategy']].to_numpy()

In [ ]:
y = y.astype(int)

y[:10]

In [ ]:
the_set = np.unique(y)

the_set

In [ ]:
len(  the_set  )

In [ ]:
_ = plt.hist(y, bins=np.arange(n_classes + 1) - 0.5, rwidth=0.8)

plt.xticks( range(n_classes), strategy_names )
plt.title("Histogram of Best_Strategy classes")

plt.show()

In [ ]:
print(X.shape)

print(y.shape)

## Train / test split

`stratify=y` keeps the same class proportions in train and test (important with only ~200 stocks).

In [ ]:
X_train, X_test, y_train, y_test, sym_train, sym_test = train_test_split(
    X, y, stock_data['Symbol'].to_numpy(),
    test_size=0.2, random_state=random_seed, stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

## Fill missing values

Missing values are filled with the **training** medians only, so no information from the test set leaks into training.

In [ ]:
train_medians = np.nanmedian( X_train.astype(float), axis=0 )

X_train = np.where( np.isnan(X_train.astype(float)), train_medians, X_train )
X_test  = np.where( np.isnan(X_test.astype(float)),  train_medians, X_test  )

print('Missing values left:', np.isnan(X_train.astype(float)).sum(), np.isnan(X_test.astype(float)).sum())

## Fix in case float64 error

In [ ]:
## fix data type

X_train = X_train.astype(  np.float32  )
X_test  = X_test.astype(   np.float32  )
y_train = y_train.astype(  np.int64 )       ## np.long
y_test  = y_test.astype(   np.int64 )

In [ ]:
X_train_tr = torch.from_numpy(X_train)
X_test_tr  = torch.from_numpy(X_test)
y_train_tr = torch.from_numpy(y_train)
y_test_tr  = torch.from_numpy(y_test)

## Normalization

* Normalization
* Standardization (*)

The means and deviations come from the training data and are stored inside each model, so the exported ONNX model accepts raw (unscaled) inputs.

In [ ]:
x_means      = X_train_tr.mean(0, keepdim=True )

x_deviations = X_train_tr.std( 0, keepdim=True) + epsilon

In [ ]:
x_means

In [ ]:
x_deviations

## Create the DataLoader

In [ ]:
X_train.shape[0]

In [ ]:
stock_train_list = [  ( X_train_tr[i],  y_train_tr[i].item()  )  for i in range( X_train.shape[0] ) ]

stock_test_list  = [  ( X_test_tr[i],   y_test_tr[i].item()   )  for i in range( X_test.shape[0]  ) ]

In [ ]:
stock_train_list[:3]

In [ ]:
train_dl = torch.utils.data.DataLoader(stock_train_list, batch_size=batch_size, shuffle=True)

In [ ]:
all_test_data = X_test.shape[0]

test_dl  = torch.utils.data.DataLoader(stock_test_list,  batch_size=all_test_data, shuffle=False)

## Neural Network Architectures

In [ ]:
## MLP - baseline (1 hidden layer)

class MLP_Net(nn.Module):
    ## init the class
    def __init__(self, x_means, x_deviations):
        super().__init__()

        self.x_means      = x_means
        self.x_deviations = x_deviations

        self.linear1 = nn.Linear(n_features, 8)
        self.act1    = nn.ReLU()   ## nn.Sigmoid()
        self.linear2 = nn.Linear(8, n_classes)
        self.act2    = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(0.25)

    ## perform inference
    def forward(self, x):

        x      = (x - self.x_means) / self.x_deviations   ## features have very different scales

        x      = self.linear1(x)
        x      = self.act1(x)
        ## x      = self.dropout(x)
        x      = self.linear2(x)
        y_pred = self.act2(x)

        return y_pred

In [ ]:
## Deep Learning with 2 hidden layers

class DL_Net(nn.Module):

    def __init__(self, x_means, x_deviations):
        super().__init__()

        self.x_means      = x_means
        self.x_deviations = x_deviations

        self.linear1 = nn.Linear(n_features, 15)
        self.act1    = nn.ReLU()
        self.linear2 = nn.Linear(15, 9)
        self.act2    = nn.ReLU()
        self.linear3 = nn.Linear(9, n_classes)
        self.act3    = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(0.25)

    ## perform inference
    def forward(self, x):

        x      = (x - self.x_means) / self.x_deviations

        x      = self.linear1(x)
        x      = self.act1(x)
        x      = self.dropout(x)
        x      = self.linear2(x)
        x      = self.act2(x)
        x      = self.dropout(x)
        x      = self.linear3(x)
        y_pred = self.act3(x)

        return y_pred

## Training Loop

In [ ]:
def training_loop( N_Epochs, model, loss_fn, opt  ):

    model.train()

    for epoch in range(N_Epochs):
        for xb, yb in train_dl:

            y_pred = model(xb)
            loss   = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % 50 == 0:
            print(epoch, "loss=", loss)

## Evaluation functions

In [ ]:
def print_metrics_function(y_test, y_pred):
    print('Accuracy: %.2f' % accuracy_score(y_test, y_pred))
    confmat = confusion_matrix(y_true=y_test, y_pred=y_pred, labels=list(range(n_classes)))
    print("Confusion Matrix:")
    print(confmat)
    print('Precision: %.3f' % precision_score(y_true=y_test, y_pred=y_pred, average='weighted', zero_division=0))
    print('Recall: %.3f' % recall_score(y_true=y_test, y_pred=y_pred, average='weighted', zero_division=0))
    print('F1-measure: %.3f' % f1_score(y_true=y_test, y_pred=y_pred, average='weighted', zero_division=0))

In [ ]:
## store the results of every model so they can be compared at the end
results = {}

def evaluate_torch_model(model, name):

    model.eval()   ## turns dropout off for testing

    with torch.no_grad():
        for x_real, y_real in test_dl:
            y_pred = model(  x_real  )
            vals, indeces = torch.max( y_pred, dim=1  )
            preds = indeces          ## integer class 0, 1 or 2 (not a probability)
            print('Integer predictions:', preds.tolist())
            print_metrics_function(y_real, preds)

    results[name] = {
        'Accuracy':  accuracy_score(y_real, preds),
        'Precision': precision_score(y_real, preds, average='weighted', zero_division=0),
        'Recall':    recall_score(y_real, preds, average='weighted', zero_division=0),
        'F1':        f1_score(y_real, preds, average='weighted', zero_division=0),
    }
    return preds

## Core Functions for MLP

In [ ]:
model_mlp  = MLP_Net( x_means, x_deviations  )

opt        = torch.optim.Adam(    model_mlp.parameters(), lr=learning_rate )

loss_fn    = nn.CrossEntropyLoss( )

training_loop(  N_Epochs, model_mlp, loss_fn, opt  )

In [ ]:
preds_mlp = evaluate_torch_model( model_mlp, 'MLP (PyTorch)' )

## Core Functions for Deep Learning

In [ ]:
model_dl   = DL_Net( x_means, x_deviations  )

opt        = torch.optim.Adam(    model_dl.parameters(), lr=learning_rate )

loss_fn    = nn.CrossEntropyLoss( )

training_loop(  N_Epochs, model_dl, loss_fn, opt  )

In [ ]:
preds_dl = evaluate_torch_model( model_dl, 'DL 2 hidden layers (PyTorch)' )

## Evaluate Model (prediction vs. real for each test stock)

Same check as the end of the regression example: compare the prediction with the real label for every test sample.

In [ ]:
model_dl.eval()

with torch.no_grad():
    y_pred_test = model_dl( X_test_tr )

y_pred_test.shape

In [ ]:
list_preds = []
list_reals = []

for i in range(len(X_test_tr)):
    np_real = y_test_tr[i].item()
    vals, indeces = torch.max( y_pred_test[i:i+1], dim=1 )   ## same as the Iris example
    np_pred = indeces.item()                                ## integer class
    print("************************************")
    print(sym_test[i], " pred, real")
    print(( np_pred, np_real ), "  ->  ", ( strategy_names[np_pred], strategy_names[np_real] ))
    list_preds.append(np_pred)
    list_reals.append(np_real)

In [ ]:
print('Correct predictions: %d / %d' % ( np.sum(np.array(list_preds) == np.array(list_reals)), len(list_reals) ))

## XGBoost

In [ ]:
classifier = xgb.XGBClassifier(

        n_estimators=100,
        reg_lambda=1,
        gamma=0,
        max_depth=3,
        random_state=random_seed
)

In [ ]:
classifier.fit(X_train, y_train.ravel())

In [ ]:
y_pred_xgb = classifier.predict(X_test)

y_pred_xgb.shape

In [ ]:
print_metrics_function(y_test.ravel(), y_pred_xgb)

In [ ]:
results['XGBoost'] = {
    'Accuracy':  accuracy_score(y_test.ravel(), y_pred_xgb),
    'Precision': precision_score(y_test.ravel(), y_pred_xgb, average='weighted', zero_division=0),
    'Recall':    recall_score(y_test.ravel(), y_pred_xgb, average='weighted', zero_division=0),
    'F1':        f1_score(y_test.ravel(), y_pred_xgb, average='weighted', zero_division=0),
}

In [ ]:
## which features does XGBoost rely on most?
pd.Series( classifier.feature_importances_, index=feature_columns ).sort_values().plot.barh( figsize=(8,5) )
plt.title('XGBoost feature importance')
plt.show()

## Compare the models

In [ ]:
results_df = pd.DataFrame(results).T.round(3)

results_df

In [ ]:
results_df[['Accuracy', 'F1']].plot.bar( figsize=(10,5), rot=0 )
plt.title('Model comparison on the test set')
plt.ylim(0, 1)
plt.show()

## Deploy PyTorch Model

Export the deep learning model to ONNX. The input is the 10 raw features (normalization is inside the model); the network output is converted to an **integer class** with argmax (`torch.max(..., dim=1)`), both here and in `index.js` on the web page.

In [ ]:
os.makedirs('deploy', exist_ok=True)

model_dl.eval()

dummy_input = torch.randn(1, n_features)

input_names  = ["input1"]
output_names = ["output1"]

export_args = dict(
        verbose=False,
        opset_version = 15,       ## opset 15 as recommended in class (newer defaults can fail on the website)
        input_names = input_names,
        output_names = output_names,
        dynamic_axes = {"input1": {0: "batch"}, "output1": {0: "batch"}}
)

try:
    torch.onnx.export(model_dl, dummy_input, "deploy/DLnet_StockStrategy.onnx", dynamo=False, **export_args)
except TypeError:      ## older PyTorch versions do not have the dynamo argument
    torch.onnx.export(model_dl, dummy_input, "deploy/DLnet_StockStrategy.onnx", **export_args)

## Validate the PyTorch ONNX model

In [ ]:
sess = rt.InferenceSession('deploy/DLnet_StockStrategy.onnx')

print('opset:', onnx.load('deploy/DLnet_StockStrategy.onnx').opset_import)

input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name

print(input_name, label_name)

In [ ]:
pred_ort = sess.run(  [label_name], {input_name: X_test.astype(np.float32)}  )

pred_ort[0][:5]

In [ ]:
preds_onnx = np.argmax( pred_ort[0], axis=1 )     ## integer class, same as torch.max(..., dim=1)

print('Integer predictions (ONNX):', preds_onnx.tolist())

print('Max difference PyTorch vs ONNX probabilities:', np.abs( pred_ort[0] - y_pred_test.numpy() ).max())
print('Same predicted class for all test stocks:', np.array_equal( preds_onnx, np.array(list_preds) ))

print_metrics_function(y_test.ravel(), preds_onnx)

## Deploy XGBoost Model

* ONNX and GitHub

In [ ]:
initial_types = [(
          'float_input',
          FloatTensorType(  [None, n_features ]  )
)]

In [ ]:
onnx_model = onnxmltools.convert_xgboost(classifier, initial_types=initial_types, target_opset=15)   ## opset 15

onnxmltools.utils.save_model(onnx_model, 'deploy/xgboost_StockStrategy_ort.onnx')

In [ ]:
sess_xgb = rt.InferenceSession('deploy/xgboost_StockStrategy_ort.onnx')

print('opset:', onnx.load('deploy/xgboost_StockStrategy_ort.onnx').opset_import)

input_name_xgb = sess_xgb.get_inputs()[0].name
label_name_xgb = sess_xgb.get_outputs()[0].name

print(input_name_xgb, label_name_xgb)

In [ ]:
pred_ort_xgb = sess_xgb.run(  [label_name_xgb], {input_name_xgb: X_test.astype(np.float32)}  )

print('Same predicted class as XGBoost for all test stocks:', np.array_equal( pred_ort_xgb[0], y_pred_xgb ))

## Save the label maps for the web page

The GitHub Pages site reads this file to fill the Sector / Industry drop-downs and to name the predicted strategy.

In [ ]:
example = X_test[0].tolist()

site_info = {
    'feature_columns': feature_columns,
    'sector_names':    sector_names,
    'industry_names':  industry_names,
    'strategy_names':  strategy_names,
    'example_symbol':  str(sym_test[0]),
    'example_values':  example,
}

with open('deploy/site_info.json', 'w') as f:
    json.dump(site_info, f, indent=2)

os.listdir('deploy')

In [ ]:
## copy the deployment files to Google Drive (next to the data)
import shutil
shutil.copytree('deploy', '/content/drive/MyDrive/HW3/deploy', dirs_exist_ok=True)

## Analysis

*Fill in after running with your data:*

* Which model has the best F1-measure, and by how much?
* Which classes are most often confused (see confusion matrices)?
* Which features are most correlated with `Best_Strategy` / most important for XGBoost?
* Limitations: only ~200 stocks, so test results vary with the split; `Sector`/`Industry` are label-encoded IDs.